# Human quality gradings

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from itertools import combinations
from pathlib import Path

# Run in parent dir
cwd = Path.cwd().resolve()
if cwd.name == "notebooks":
    os.chdir(cwd.parent)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import spearmanr

from nako_eye.utils import helpers, nako, plot

device = 'cuda:0'
pd.options.display.float_format = "{:,.2f}".format
plot.set_rc_params(kind='paper', notebook_dpi=120)

## Join gradings from different models with human gradings

In [ ]:
data_path = Path('data')
graders = ['berens', 'beuse', 'brandl', 'seitz', 'zeymer']
quality_map_human = {
    'Unusable (Reject)': 0,
    'Poor': 1,
    'Fair (Usable)': 2,
    'Good': 3,
    'Excellent': 4,
}
quality_labels_human = {v: k for k, v in quality_map_human.items()}

1. Add gradings from other models.

In [ ]:
model_name = 'retiratey'
# cols = {'prediction_label': f'{model_name} class', 'label': f'{model_name} label'}
cols = {'label': f'{model_name} label'}

# Load quality grading from 100 images
df_quality = pd.read_csv(data_path / 'quality_study' / 'quality_grading.csv')
df_quality['ID'] = df_quality['ID'].astype(str)

dfs = []
for image_type in nako.IMAGE_TYPES:
    for idx in range(1, 4):
        df_model = pd.read_csv(f'./data/quality_results/590_baseline/{model_name}/{image_type}_{idx}.csv')
        df_model['type'] = image_type
        df_model['index'] = idx
        df_model = df_model.rename(columns=cols)
        dfs.append(df_model[['ID', 'type', 'index'] + list(cols.values())])

df_predictions = pd.concat(dfs, ignore_index=True)
df_predictions['ID'] = df_predictions['ID'].astype(str)

df_quality = df_quality.merge(df_predictions, on=['ID', 'type', 'index'], how='left')
df_quality

# df_quality.to_csv(data_path / 'quality_study' / 'quality_grading.csv', index=False)

2. Add ratings from human graders.

In [ ]:
df_quality = pd.read_csv(data_path / 'quality_study' / 'quality_grading.csv')
df_quality['ID'] = df_quality['ID'].astype(str)
df_quality = df_quality.sort_values(by='ID').reset_index(drop=True)
df_quality['image_number'] = df_quality.index + 1  # matches "Image Number" in the grader xlsx files

for grader in graders:
    df_grader = pd.read_excel(data_path / 'quality_study' / f'quality_grading - {grader}.xlsx')
    df_grader = df_grader.dropna(subset=['Rating'])
    df_quality[f'{grader}'] = df_quality['image_number'].map(
        dict(zip(df_grader['Image Number'], df_grader['Rating'].map(quality_map_human)))
    )

# Create average and majority grader
df_quality['average'] = df_quality[graders].mean(axis=1)
df_quality['average (rounded)'] = np.floor(df_quality['average'] + 0.5).astype(int)
df_quality['majority'] = [helpers.majority_vote(df_quality.loc[i, graders], df_quality.loc[i, 'average (rounded)']) for i in df_quality.index]
df_quality

# df_quality.to_csv(data_path / 'quality_study' / 'quality_grading.csv', index=False)

## Inter-rater agreement: human gradings on a 5-level scale

In [ ]:
# Load model scores and human gradings for 100 images
df_quality = pd.read_csv(data_path / 'quality_study' / 'quality_grading.csv')
df_quality.head()

In [ ]:
# Quadratic Weighted Kappa
graders_discrete = graders + ['average (rounded)', 'majority']
kappa_results = []
for g1, g2 in combinations(graders_discrete, 2):
    kappa, p = helpers.kappa_test(df_quality[g1], df_quality[g2], weights='quadratic')
    kappa_results.append({'rater_1': g1, 'rater_2': g2, 'kappa': kappa, 'p': p})
kappa_df = pd.DataFrame(kappa_results)

kappa_matrix = pd.DataFrame(np.eye(len(graders_discrete)), index=graders_discrete, columns=graders_discrete)
kappa_pval_matrix = pd.DataFrame(np.zeros((len(graders_discrete), len(graders_discrete))), index=graders_discrete, columns=graders_discrete)
for _, row in kappa_df.iterrows():
    kappa_matrix.loc[row['rater_1'], row['rater_2']] = row['kappa']
    kappa_matrix.loc[row['rater_2'], row['rater_1']] = row['kappa']
    kappa_pval_matrix.loc[row['rater_1'], row['rater_2']] = row['p']
    kappa_pval_matrix.loc[row['rater_2'], row['rater_1']] = row['p']

# Spearman's correlation
graders_all = graders + ['average', 'majority']
spearman_results = []
for g1, g2 in combinations(graders_all, 2):
    rho, p = spearmanr(df_quality[g1], df_quality[g2])
    spearman_results.append({'rater_1': g1, 'rater_2': g2, 'spearman_rho': rho, 'p': p})
spearman_df = pd.DataFrame(spearman_results)

spearman_matrix = pd.DataFrame(np.eye(len(graders_all)), index=graders_all, columns=graders_all)
spearman_pval_matrix = pd.DataFrame(np.zeros((len(graders_all), len(graders_all))), index=graders_all, columns=graders_all)
for _, row in spearman_df.iterrows():
    spearman_matrix.loc[row['rater_1'], row['rater_2']] = row['spearman_rho']
    spearman_matrix.loc[row['rater_2'], row['rater_1']] = row['spearman_rho']
    spearman_pval_matrix.loc[row['rater_1'], row['rater_2']] = row['p']
    spearman_pval_matrix.loc[row['rater_2'], row['rater_1']] = row['p']

In [ ]:
fig, axes = plt.subplots(1, 2)
plot.set_figsize(fig, 8, height_ratio=0.45)

sns.heatmap(kappa_matrix, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1, square=True, linewidths=0.5, cbar_kws={'shrink': 0.4}, ax=axes[0])
axes[0].set_title('Quadratic Weighted Kappa', pad=5)
axes[0].tick_params(rotation=0)
axes[0].set_yticklabels(axes[0].get_yticklabels(), rotation=0)
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')

sns.heatmap(spearman_matrix, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1, square=True, linewidths=0.5, cbar_kws={'shrink': 0.4}, ax=axes[1])
axes[1].set_title("Spearman's correlation", pad=5)
axes[1].tick_params(rotation=0)
axes[1].set_yticklabels(axes[1].get_yticklabels(), rotation=0)
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()

## Inter-rater agreement: human gradings on a binary scale

In [ ]:
# Cohen's Kappa
graders_discrete = graders + ['average (rounded)', 'majority']
kappa_results = []
for g1, g2 in combinations(graders_discrete, 2):
    kappa, p = helpers.kappa_test((df_quality[g1] >= 2).astype(int), (df_quality[g2] >= 2).astype(int))
    kappa_results.append({'rater_1': g1, 'rater_2': g2, 'kappa': kappa, 'p': p})
kappa_df = pd.DataFrame(kappa_results)

kappa_matrix = pd.DataFrame(np.eye(len(graders_discrete)), index=graders_discrete, columns=graders_discrete)
kappa_pval_matrix = pd.DataFrame(np.zeros((len(graders_discrete), len(graders_discrete))), index=graders_discrete, columns=graders_discrete)
for _, row in kappa_df.iterrows():
    kappa_matrix.loc[row['rater_1'], row['rater_2']] = row['kappa']
    kappa_matrix.loc[row['rater_2'], row['rater_1']] = row['kappa']
    kappa_pval_matrix.loc[row['rater_1'], row['rater_2']] = row['p']
    kappa_pval_matrix.loc[row['rater_2'], row['rater_1']] = row['p']

# Spearman's correlation
graders_all = graders + ['average', 'majority']
spearman_results = []
for g1, g2 in combinations(graders_all, 2):
    rho, p = spearmanr((df_quality[g1] >= 2).astype(int), (df_quality[g2] >= 2).astype(int))
    spearman_results.append({'rater_1': g1, 'rater_2': g2, 'spearman_rho': rho, 'p': p})
spearman_df = pd.DataFrame(spearman_results)

spearman_matrix = pd.DataFrame(np.eye(len(graders_all)), index=graders_all, columns=graders_all)
spearman_pval_matrix = pd.DataFrame(np.zeros((len(graders_all), len(graders_all))), index=graders_all, columns=graders_all)
for _, row in spearman_df.iterrows():
    spearman_matrix.loc[row['rater_1'], row['rater_2']] = row['spearman_rho']
    spearman_matrix.loc[row['rater_2'], row['rater_1']] = row['spearman_rho']
    spearman_pval_matrix.loc[row['rater_1'], row['rater_2']] = row['p']
    spearman_pval_matrix.loc[row['rater_2'], row['rater_1']] = row['p']

In [ ]:
fig, axes = plt.subplots(1, 2)
plot.set_figsize(fig, 8, height_ratio=0.45)

sns.heatmap(kappa_matrix, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1, square=True, linewidths=0.5, cbar_kws={'shrink': 0.5}, ax=axes[0])
axes[0].set_title("Cohen's Kappa", pad=5)
axes[0].tick_params(rotation=0)
axes[0].set_yticklabels(axes[0].get_yticklabels(), rotation=0)
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')

sns.heatmap(spearman_matrix, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1, square=True, linewidths=0.5, cbar_kws={'shrink': 0.5}, ax=axes[1])
axes[1].set_title("Spearman's correlation", pad=5)
axes[1].tick_params(rotation=0)
axes[1].set_yticklabels(axes[1].get_yticklabels(), rotation=0)
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')

plt.tight_layout(w_pad=3)

In [ ]:
print("Cohen's Kappa")
print(kappa_pval_matrix.to_csv(sep='\t', float_format='%.2e'))
print("Spearman's Correlation")
print(spearman_pval_matrix.to_csv(sep='\t', float_format='%.2e'))

## Model 1: Fundus Image Toolbox

In [ ]:
# Model confidence (continuous) vs. human rating (discrete)
graders_all = graders + ['average', 'majority']
tick_labels = [quality_labels_human[k].split()[0] for k in sorted(quality_labels_human)]

fig, axes = plt.subplots(1, len(graders_all), sharey=True)
plot.set_figsize(fig, 14, height_ratio=0.2)

for ax, grader in zip(axes, graders_all):
    if grader == 'average':
       sns.regplot(data=df_quality, x=grader, y='FIT score', ax=ax, scatter_kws=dict(color='black', alpha=0.6, s=5))
       ax.axhline(0.25, color='red', linestyle='--', lw=1)
       ax.set_ylabel('')
       plot.detach_axes(ax)
    else:
        sns.boxplot(data=df_quality, x=grader, y='FIT score', color='lightgray', showfliers=False, ax=ax)
        sns.stripplot(data=df_quality, x=grader, y='FIT score', color='black', alpha=0.6, size=3, ax=ax)
        ax.axhline(0.25, color='red', linestyle='--', lw=1)
        plot.detach_axes(ax)
        ax.set_xticks(sorted(quality_labels_human), labels=tick_labels, rotation=45, ha='right')

plot.set_labs(axes, xlabs='Human rating', titles=[g.capitalize() for g in graders_all])
axes[0].set_ylabel('FIT score')
plt.tight_layout()

In [ ]:
# Average of the 5 SDs and IQRs for each grade
def iqr(x):
    return np.percentile(x, 75) - np.percentile(x, 25)

spread_results = []
for grader in graders:  # ['berens', 'beuse', 'brandl']
    grouped = df_quality.groupby(grader)['FIT score']
    for grade, vals in grouped:
        spread_results.append({'grader': grader, 'grade': grade, 'std': vals.std(), 'iqr': iqr(vals)})

spread_df = pd.DataFrame(spread_results)

# Pivot so rows = grade, columns = grader, one table per metric
std_table = spread_df.pivot(index='grade', columns='grader', values='std')
iqr_table = spread_df.pivot(index='grade', columns='grader', values='iqr')

# Add the average across raters as an extra column
std_table['average'] = std_table[graders].mean(axis=1)
iqr_table['average'] = iqr_table[graders].mean(axis=1)

std_table.index = std_table.index.map(quality_labels_human)
iqr_table.index = iqr_table.index.map(quality_labels_human)

print("Std of FIT score per grade")
display(std_table)

print("IQR of FIT score per grade")
display(iqr_table)


In [ ]:
# Model vs graders agreement
graders_bin = graders + ['average (rounded)', 'majority']
kappa_results = []
for grader in graders_bin:
    kappa, p = helpers.kappa_test(df_quality['FIT label'], (df_quality[grader] >= 2).astype(int))
    kappa_results.append({'grader': grader, 'kappa': kappa, 'p': p})
kappa_df = pd.DataFrame(kappa_results).set_index('grader')

graders_all = graders + ['average', 'majority']
spearman_results = []
for grader in graders_all:
    rho, p = spearmanr(df_quality['FIT score'], df_quality[grader])
    spearman_results.append({'grader': grader, 'spearman_rho': rho, 'p': p})
spearman_df = pd.DataFrame(spearman_results).set_index('grader')

In [ ]:
def p_label(p):
    return f'p={p:.1e}'

fig, axes = plt.subplots(1, 2)
plot.set_figsize(fig, 'full', height_ratio=0.45)

bar_labels = [g.capitalize() for g in spearman_df.index]
axes[0].barh(bar_labels, spearman_df['spearman_rho'], color='steelblue')
for i, (rho, p) in enumerate(zip(spearman_df['spearman_rho'], spearman_df['p'])):
    axes[0].text(rho + 0.02, i, p_label(p), ha='left', va='center', fontsize=7)
axes[0].set_xlim(0, 1)
axes[0].invert_yaxis()
axes[0].set_xlabel("Spearman's correlation")
axes[0].set_title('FIT score vs. graders (5-level)', pad=5)
plot.detach_axes(axes[0])

bar_labels = [g.capitalize() for g in kappa_df.index]
axes[1].barh(bar_labels, kappa_df['kappa'], color='steelblue')
for i, (kappa, p) in enumerate(zip(kappa_df['kappa'], kappa_df['p'])):
    axes[1].text(kappa + 0.02, i, p_label(p), ha='left', va='center', fontsize=7)
axes[1].set_xlim(0, 1)
axes[1].invert_yaxis()
axes[1].set_xlabel("Cohen's Kappa")
axes[1].set_title('FIT vs. graders (binary)', pad=5)
plot.detach_axes(axes[1])

plt.tight_layout(w_pad=3)

### Fundus Image Toolbox on training dataset (DeepDriD)

In [ ]:
# Load model scores and human gradings for 100 images
df_deepdrid = pd.read_csv(data_path / 'quality_study' / 'quality_grading_deepdrid.csv')
df_deepdrid['image_id'] = df_deepdrid['image_path'].map(lambda x: x.split(sep='/')[-1].split(sep='.')[0])
df_deepdrid_gt = pd.read_csv(data_path / 'quality_study' / 'deepdrid_gt.csv')
df_deepdrid = df_deepdrid.merge(df_deepdrid_gt, on=['image_id'], how='left')
df_deepdrid.head()

In [ ]:
# Model vs ground truth agreement
kappa, kappa_p = helpers.kappa_test(df_deepdrid['label'], df_deepdrid['Overall quality'])
rho, rho_p = spearmanr(df_deepdrid['label'], df_deepdrid['Overall quality'])

print(f"Cohen's Kappa: {kappa:.3f} ({p_label(kappa_p)})")
print(f"Spearman's correlation: {rho:.3f} ({p_label(rho_p)})")

## Model 2: AutoMorph

In [ ]:
# Model vs graders agreement
quality_map_automorph = {'Reject': 0, 'Usable': 1, 'Good': 2}
class_ordinal = df_quality['automorph class'].map(quality_map_automorph)

graders_bin = graders + ['average (rounded)', 'majority']
kappa_results = []
for grader in graders_bin:
    kappa, p = helpers.kappa_test(df_quality['automorph label'], (df_quality[grader] >= 2).astype(int))
    kappa_results.append({'grader': grader, 'kappa': kappa, 'p': p})
kappa_df = pd.DataFrame(kappa_results).set_index('grader')

graders_all = graders + ['average', 'majority']
spearman_results = []
for grader in graders_all:
    rho, p = spearmanr(class_ordinal, df_quality[grader])
    spearman_results.append({'grader': grader, 'spearman_rho': rho, 'p': p})
spearman_df = pd.DataFrame(spearman_results).set_index('grader')

In [ ]:
fig, axes = plt.subplots(1, 2)
plot.set_figsize(fig, 'full', height_ratio=0.45)

bar_labels = [g.capitalize() for g in spearman_df.index]
axes[0].barh(bar_labels, spearman_df['spearman_rho'], color='steelblue')
for i, (rho, p) in enumerate(zip(spearman_df['spearman_rho'], spearman_df['p'])):
    axes[0].text(rho + 0.02, i, p_label(p), ha='left', va='center', fontsize=7)
axes[0].set_xlim(0, 1)
axes[0].invert_yaxis()
axes[0].set_xlabel("Spearman's correlation")
axes[0].set_title('AutoMorph classes (3-level) vs. graders (5-level)', pad=5)
plot.detach_axes(axes[0])

bar_labels = [g.capitalize() for g in kappa_df.index]
axes[1].barh(bar_labels, kappa_df['kappa'], color='steelblue')
for i, (kappa, p) in enumerate(zip(kappa_df['kappa'], kappa_df['p'])):
    axes[1].text(kappa + 0.02, i, p_label(p), ha='left', va='center', fontsize=7)
axes[1].set_xlim(0, 1)
axes[1].invert_yaxis()
axes[1].set_xlabel("Cohen's Kappa")
axes[1].set_title('AutoMorph vs. graders (binary)', pad=5)
plot.detach_axes(axes[1])

plt.tight_layout(w_pad=3)

## Model 3: VascX

In [ ]:
# Model vs graders agreement
quality_map_vascx = {'Reject': 0, 'Usable': 1, 'Good': 2}
class_ordinal = df_quality['vascx class'].map(quality_map_vascx)

graders_bin = graders + ['average (rounded)', 'majority']
kappa_results = []
for grader in graders_bin:
    kappa, p = helpers.kappa_test(df_quality['vascx label'], (df_quality[grader] >= 2).astype(int))
    kappa_results.append({'grader': grader, 'kappa': kappa, 'p': p})
kappa_df = pd.DataFrame(kappa_results).set_index('grader')

graders_all = graders + ['average', 'majority']
spearman_results = []
for grader in graders_all:
    rho, p = spearmanr(class_ordinal, df_quality[grader])
    spearman_results.append({'grader': grader, 'spearman_rho': rho, 'p': p})
spearman_df = pd.DataFrame(spearman_results).set_index('grader')

In [ ]:
fig, axes = plt.subplots(1, 2)
plot.set_figsize(fig, 'full', height_ratio=0.45)

bar_labels = [g.capitalize() for g in spearman_df.index]
axes[0].barh(bar_labels, spearman_df['spearman_rho'], color='steelblue')
for i, (rho, p) in enumerate(zip(spearman_df['spearman_rho'], spearman_df['p'])):
    axes[0].text(rho + 0.02, i, p_label(p), ha='left', va='center', fontsize=7)
axes[0].set_xlim(0, 1)
axes[0].invert_yaxis()
axes[0].set_xlabel("Spearman's correlation")
axes[0].set_title('VascX classes (3-level) vs. graders (5-level)', pad=5)
plot.detach_axes(axes[0])

bar_labels = [g.capitalize() for g in kappa_df.index]
axes[1].barh(bar_labels, kappa_df['kappa'], color='steelblue')
for i, (kappa, p) in enumerate(zip(kappa_df['kappa'], kappa_df['p'])):
    axes[1].text(kappa + 0.02, i, p_label(p), ha='left', va='center', fontsize=7)
axes[1].set_xlim(0, 1)
axes[1].invert_yaxis()
axes[1].set_xlabel("Cohen's Kappa")
axes[1].set_title('VascX vs. graders (binary)', pad=5)
plot.detach_axes(axes[1])

plt.tight_layout(w_pad=3)

### Model 4: RetiRatey

In [ ]:
# Model vs graders agreement
graders_bin = graders + ['average (rounded)', 'majority']
kappa_results = []
for grader in graders_bin:
    kappa, p = helpers.kappa_test(df_quality['retiratey label'], (df_quality[grader] >= 2).astype(int))
    kappa_results.append({'grader': grader, 'kappa': kappa, 'p': p})
kappa_df = pd.DataFrame(kappa_results).set_index('grader')

In [ ]:
fig, ax = plt.subplots(1, 1)
plot.set_figsize(fig, 'col', height_ratio=0.8)

bar_labels = [g.capitalize() for g in kappa_df.index]
ax.barh(bar_labels, kappa_df['kappa'], color='steelblue')
for i, (kappa, p) in enumerate(zip(kappa_df['kappa'], kappa_df['p'])):
    ax.text(kappa + 0.02, i, p_label(p), ha='left', va='center', fontsize=7)
ax.set_xlim(0, 1)
ax.invert_yaxis()
ax.set_xlabel("Cohen's Kappa")
ax.set_title('RetiRatey vs. graders (binary)', pad=5)
plot.detach_axes(ax)

plt.tight_layout(w_pad=3)